In [ ]:
# ========== Selenium 网页抓取 → OpenAI 摘要 ==========
# 练习目标：用真实浏览器渲染动态页，抽正文，再交给 gpt-4o-mini 做友好摘要
# 和本课关系：Website 抓取（比 requests 更能跑 JS）+ Chat Completions + Notebook Markdown 展示
# 怎么跑：准备 .env 的 OPENAI_API_KEY；本机有 Chrome；从上到下运行本格

# 导入标准库 time：滚动加载时 sleep 等待动态内容
import time
# 导入标准库 os：读环境变量（Environment Variables）里的 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程，避免写进代码
from dotenv import load_dotenv
# 导入标准库 logging：打印抓取进度（Loading / Scrolled / Extracted）
import logging
# 从 selenium 导入 webdriver：启动并控制真实 Chrome 浏览器
from selenium import webdriver
# Service：指定 ChromeDriver 可执行文件的启动方式
from selenium.webdriver.chrome.service import Service
# Options：给 Chrome 传无头、窗口大小、user-agent 等启动参数
from selenium.webdriver.chrome.options import Options
# ChromeDriverManager：缺驱动时自动下载匹配版本的 ChromeDriver
from webdriver_manager.chrome import ChromeDriverManager
# BeautifulSoup：从渲染后的 HTML 里抽纯文本、去掉 script/style 等噪声标签
from bs4 import BeautifulSoup
# IPython 展示：把模型摘要渲染成 Markdown
from IPython.display import Markdown, display
# OpenAI 客户端：调用云端 Chat Completions API
from openai import OpenAI

# 理念说明：下面这段三引号字符串是「文档用」表达式（接在 import 后，不是模块首行 docstring）
# 为什么用 Selenium / 本程序做什么 —— 保留原英文，避免改可执行字符串字面量；中文要点见上行与各段旁注
"""
This program loads any webpage using Selenium, extracts its readable text content,
and summarizes it using an OpenAI model.

WHY SELENIUM?
-------------
- Selenium controls a real browser (Chrome), which allows full JavaScript rendering.
- Works reliably on Windows, UV, and Jupyter/VSCode Notebook environments.
- Can handle dynamic content, infinite scrolling, AJAX, React/Vue/Next.js sites.
- Unlike Playwright, it has no issues with subprocess creation on Windows/Jupyter.

WHAT THIS PROGRAM DOES:
-----------------------
1. Opens a webpage in a real headless Chrome browser.
2. Loads dynamic content by automatically scrolling.
3. Extracts visible text while removing scripts/images/styles.
4. Sends cleaned content to an OpenAI model for summarization.
5. Displays the summary in Markdown format.

In short, Selenium solves the rendering problem, and OpenAI handles the NLP.
"""

# ---------- 环境：加载密钥并做硬失败检查 ----------
# override=True：.env 里的值覆盖已有环境变量
load_dotenv(override=True)
# 读取 OpenAI API Key（变量名必须是 OPENAI_API_KEY）
api_key = os.getenv("OPENAI_API_KEY")

# 没有密钥就立刻抛错，避免后面请求才失败（错误文案保持英文原样）
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found. Please set it in your environment.")

# 创建 OpenAI 客户端（默认会用环境里的 OPENAI_API_KEY）
openai = OpenAI()

# ---------- 日志：INFO 级别 + 自定义格式前缀 ----------
logging.basicConfig(
    level=logging.INFO,
    format="🟦 [%(levelname)s] %(message)s"
)


class WebsiteError(Exception):
    """网站抓取相关的自定义异常，便于和外层其它错误区分。"""
    pass


class Website:
    """封装「一个 URL」：用 Selenium 打开、滚动、抽 title/text。"""

    def __init__(self, url):
        # 保存目标 URL；title/text 稍后在 initialize() 里填充
        self.url = url
        self.title = None
        self.text = None

    @classmethod
    def create(cls, url):
        """工厂方法：实例化后立刻 initialize，返回已填好正文的 Website。"""
        website = cls(url)
        website.initialize()
        return website

    def initialize(self):
        """打开网页、自动滚动加载动态内容，再清洗出可读正文。"""
        # 打日志：开始加载（消息字符串保持原样）
        logging.info(f"🌐 Loading webpage: {self.url}")

        # ---------- Chrome 启动参数：无头 + 稳定性常见旗标 ----------
        options = Options()
        # 新版无头模式（不弹窗）
        options.add_argument("--headless=new")
        # 无 GPU（服务器/CI 常见）
        options.add_argument("--disable-gpu")
        # 禁用沙箱（部分 Linux/容器环境需要）
        options.add_argument("--no-sandbox")
        # 避免 /dev/shm 过小导致 Chrome 崩溃
        options.add_argument("--disable-dev-shm-usage")
        # 视口大小，影响部分响应式站点渲染
        options.add_argument("--window-size=1920,1080")
        # 降低「自动化受控」特征（部分站点会检测）
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_argument("start-maximized")
        # 伪装成常见桌面 Chrome 的 User-Agent 字符串（保持原样）
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        )

        # 创建 Chrome WebDriver；ChromeDriverManager 会按需安装驱动
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )

        try:
            # 导航到目标 URL
            driver.get(self.url)
            # 先等 2 秒，给首屏 JS 一点时间
            time.sleep(2)

            # ---------- 自动滚动：触发懒加载 / 无限滚动内容 ----------
            # 每次滚动后的停顿秒数
            scroll_pause = 1.0
            # 最多滚几轮（可调；太大浪费时间）
            max_scrolls = 6   # adjustable

            # 用 JS 读当前文档高度，作为「是否还在变长」的基准
            last_height = driver.execute_script("return document.body.scrollHeight")

            for i in range(max_scrolls):
                # 滚到页面底部，触发更多内容加载
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(scroll_pause)
                # 再量一次高度
                new_height = driver.execute_script("return document.body.scrollHeight")

                # 高度不再增加 → 认为已到底，提前结束
                if new_height == last_height:
                    logging.info("✔ Reached bottom of page.")
                    break

                last_height = new_height
                logging.info(f"Scrolled page ({i+1}/{max_scrolls})")

            # ---------- 抽取标题与正文 ----------
            # driver.title 为空时用占位标题（字符串保持原样）
            self.title = driver.title or "Untitled Page"
            logging.info(f"📝 Extracted title: {self.title}")

            # 取浏览器当前 DOM 序列化后的完整 HTML
            html = driver.page_source

            # 用 html.parser 解析（无需 lxml 额外依赖）
            soup = BeautifulSoup(html, "html.parser")

            # 删掉脚本/样式/图片等对「可读摘要」无用的标签
            for tag in soup.find_all(["script", "style", "noscript", "img", "svg", "input", "meta"]):
                tag.decompose()

            # 从 body 抽文本：换行分隔、去掉首尾空白；没有 body 则空串
            body = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
            # 去掉空行，得到更干净的纯文本
            clean_text = "\n".join(line for line in body.splitlines() if line.strip())

            self.text = clean_text

            logging.info("✔ Text extraction completed.")

        except Exception as e:
            # 包装成 WebsiteError；错误文案模板保持英文（含 URL）
            raise WebsiteError(f"Failed to scrape {self.url}: {e}")

        finally:
            # 无论成功失败都关掉浏览器，避免僵尸进程
            driver.quit()


# ---------- 组装发给模型的 messages（system 定语气，user 放标题+正文） ----------
def messages_for(website):
    """把 Website 的 title/text 打成 Chat Completions 的 messages 列表。"""
    return [
        # system prompt 保留英文：规定有趣、友好、准确、不人身攻击的摘要风格
        {"role": "system", "content": "Provide a fun, friendly, and respectful summary of the webpage. "
                "Use light humor and a playful tone, but stay accurate and do not insult "
                "the author, the content, or the subject. Make it enjoyable to read."},
        {
            "role": "user",
            # user 内容：标题 + 清洗后的正文（结构字符串保持英文标签 Title/Content）
            "content": f"Title: {website.title}\n\nContent:\n{website.text}"
        }
    ]


# ---------- 摘要：抓取 + 一次非流式 Chat Completions ----------
def summarize(url):
    """给定 URL：Selenium 抓取 → gpt-4o-mini 摘要 → 返回文本。"""
    # 工厂方法：内部会启动浏览器并填充 title/text
    website = Website.create(url)
    # 调用云端小模型；model id 保持 gpt-4o-mini
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(website)
    )
    # 取第一条回复的文本内容
    return response.choices[0].message.content


# ---------- 在 Notebook 里用 Markdown 展示摘要 ----------
def display_summary(url):
    """summarize 后 display(Markdown(...))，适合 Jupyter 阅读。"""
    summary = summarize(url)
    display(Markdown(summary))


# 示例：对 Udemy 首页做一轮「抓取 → 摘要 → 展示」（URL 保持原样）
display_summary("https://udemy.com")
